# Re-run Full-Cohort Events Where Text Hurts (delta c-index < 0)

Identifies every full-cohort event for which adding text embeddings **lowers** the test c-index
relative to the base model (`text_cindex - base_cindex < 0`), then re-runs each of those
`(scheme, event)` pairs individually through the same pipeline used by
`python_scripts/model_training/run_full_cohort_event.py`.

**Source of compiled metrics:** the manuscript_figures pipeline — specifically
`fig2_full_cohort_metrics.csv` written by
`jupyter_notebooks/manuscript_figures/data_generation/prep_figure_2.py`
(columns: `scheme, event, text_cindex, base_cindex, text_auc, base_auc`). This notebook does **not**
use `compile_all_scheme_results.ipynb`.

**Selection criterion:** strictly negative delta c-index (worse with text).

**Output behavior:** each re-run is non-destructive by default — the original `full_cohort/{event}/`
results (`base_test.csv`, `base_val.csv`, `text_test.csv`, `text_val.csv`) are **overwritten only
when the re-run improves the delta c-index** (`new_delta > old_delta`). When overwriting, the
originals are first copied to a timestamped `full_cohort_prererun_backup/` folder
(toggle with `BACKUP_ORIGINALS`). Events that do not improve are left untouched.

Because `prep_figure_2.py` reads `mean_c_index` straight out of `text_test.csv` / `base_test.csv`,
re-running `prep_figure_2.py` afterward will pick up any overwritten results automatically.

Pipeline parity with `run_full_cohort_event.py`:
- `filter_event_rows` -> `validate_cox_inputs` -> drop NaN feature rows
- text model: `run_grid_CoxPH_parallel` over `DEFAULT_L1_RATIOS` x `DEFAULT_ALPHAS`
- base model: `run_base_CoxPH` (age only as the unpenalized continuous covariate)

> Note: the data/results live on the cluster under `/data/gusev/...`, so run this notebook there
> (e.g. on a node with several CPUs). The grid search is the slow part.

In [ ]:
import os
import sys
import time
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# --- Locate repo root and wire up imports (robust to launch dir) ---
CWD = Path.cwd().resolve()
REPO_ROOT = None
for cand in [CWD, *CWD.parents]:
    if (cand / 'jupyter_notebooks' / 'manuscript_figures').exists() and (cand / 'python_utils').exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repo root from {CWD}')

MANUSCRIPT_DIR = REPO_ROOT / 'jupyter_notebooks' / 'manuscript_figures'
for sub in (MANUSCRIPT_DIR, REPO_ROOT / 'python_utils', REPO_ROOT / 'python_scripts' / 'model_training'):
    if str(sub) not in sys.path:
        sys.path.insert(0, str(sub))

# Compiled metrics come from the manuscript_figures pipeline (prep_figure_2.py),
# NOT compile_all_scheme_results.ipynb. Importing _figure_utils also puts
# python_scripts/model_training on sys.path and gives us the shared path helpers.
from _figure_utils import (
    FIGURE_DATA_DIR, RESULTS_PATH, SURV_PATH,
    figure_data_path, full_cohort_event_dir, scheme_results_dir,
)
from embed_surv_utils import run_base_CoxPH, run_grid_CoxPH_parallel
from slurm_array_utils import (
    DEFAULT_ALPHAS, DEFAULT_L1_RATIOS,
    build_full_prediction_df, filter_event_rows, validate_cox_inputs,
)

# Figure-2 full-cohort metrics: scheme, event, text_cindex, base_cindex, text_auc, base_auc
FIG2_METRICS_FP = figure_data_path('fig2_full_cohort_metrics.csv')

# --- Re-run knobs ---
N_JOBS = int(os.getenv('SLURM_CPUS_PER_TASK', os.cpu_count() or 1))
MAX_ITER = 1000
BACKEND = 'threading'           # matches run_full_cohort_event.py default
BACKUP_ORIGINALS = True         # copy existing csvs before overwriting
os.environ.setdefault('JOBLIB_DEFAULT_WORKER_TIMEOUT', '600')

BASE_VARS = ['GENDER', 'AGE_AT_TREATMENTSTART']
BACKUP_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'REPO_ROOT        : {REPO_ROOT}')
print(f'Fig2 metrics CSV : {FIG2_METRICS_FP}')
print(f'  exists         : {os.path.isfile(FIG2_METRICS_FP)}')
print(f'N_JOBS           : {N_JOBS}')
print(f'Backup originals : {BACKUP_ORIGINALS} (stamp {BACKUP_STAMP})')

## 1. Identify events where text hurts (delta c-index < 0)

Load `fig2_full_cohort_metrics.csv` (from the manuscript_figures `prep_figure_2.py` step) and select
every `(scheme, event)` whose full-cohort text model scored a **lower** test c-index than the base
model. `delta_c_index` is computed here as `text_cindex - base_cindex`.

> If this file is missing, run `prep_figure_2.py` first (`python -m data_generation.prep_figure_2`
> from `jupyter_notebooks/manuscript_figures/`, or via `run_all_figures.ipynb`).

In [ ]:
fig2 = pd.read_csv(FIG2_METRICS_FP)
print(f'Loaded {len(fig2)} event-scheme rows from fig2_full_cohort_metrics.csv')

for col in ('base_cindex', 'text_cindex'):
    if col not in fig2.columns:
        raise KeyError(f"Expected column '{col}' missing from {FIG2_METRICS_FP}")

# delta c-index = text - base  (negative => text performs worse than base)
fig2['delta_c_index'] = fig2['text_cindex'] - fig2['base_cindex']

# Both models must have a c-index for the comparison to be meaningful
valid = fig2.dropna(subset=['base_cindex', 'text_cindex']).copy()

# Strictly negative delta = text worse than base
negative_delta = (
    valid.loc[valid['delta_c_index'] < 0]
    .sort_values('delta_c_index')  # most-harmful first
    .reset_index(drop=True)
)

display_cols = ['scheme', 'event', 'base_cindex', 'text_cindex', 'delta_c_index']
print(f'\nEvents with delta_c_index < 0: {len(negative_delta)} / {len(valid)} comparable events')
print('\nBy scheme:')
print(negative_delta['scheme'].value_counts().to_string())
negative_delta[display_cols].round(4)

## 2. Re-run function (overwrite only if delta improves)

`rerun_if_improved` re-fits both the base and text models for one event using the exact same steps
as `run_full_cohort_event.py`, then computes the **new** delta c-index (`new_text - new_base`). It
compares this to the **old** delta from the compiled table and:

- **only overwrites** the original `full_cohort/{event}/` csvs when `new_delta > old_delta`
  (backing up the originals first when `BACKUP_ORIGINALS=True`);
- otherwise leaves the originals untouched and reports `kept_original`.

Prediction dataframes are built once per scheme and cached, so re-running many events in the same
scheme does not reload the large embedding files repeatedly.

In [ ]:
_scheme_cache = {}


def _get_scheme_data(scheme):
    """Build (and cache) the full prediction df for a scheme."""
    if scheme not in _scheme_cache:
        print(f'  [load] building prediction df for scheme={scheme} ...')
        _scheme_cache[scheme] = build_full_prediction_df(scheme)
    return _scheme_cache[scheme]


def _backup_existing(scheme, event, out_dir, filenames):
    """Copy existing result files to {scheme_results_dir}/full_cohort_prererun_backup/{stamp}/{event}/."""
    backup_dir = os.path.join(scheme_results_dir(scheme), 'full_cohort_prererun_backup', BACKUP_STAMP, event)
    copied = []
    for name in filenames:
        src = os.path.join(out_dir, name)
        if os.path.isfile(src):
            os.makedirs(backup_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(backup_dir, name))
            copied.append(name)
    return backup_dir if copied else None


def rerun_if_improved(scheme, event, old_base_c, old_text_c,
                      n_jobs=N_JOBS, max_iter=MAX_ITER, backend=BACKEND):
    """Re-fit base + text for one event; overwrite originals ONLY if delta c-index improves.

    "Improved" means new (text - base) c-index > old (text - base) c-index. When the re-run does
    not beat the prior delta, nothing is written and the original results are left in place.
    """
    old_delta = (old_text_c - old_base_c) if pd.notna(old_text_c) and pd.notna(old_base_c) else np.nan
    result = {
        'scheme': scheme, 'event': event,
        'old_base_c_index': old_base_c, 'old_text_c_index': old_text_c,
        'old_delta_c_index': old_delta,
        'new_base_c_index': np.nan, 'new_text_c_index': np.nan, 'new_delta_c_index': np.nan,
        'improved': False, 'backup_dir': None,
        'text_minutes': np.nan, 'base_minutes': np.nan,
    }

    full_df, type_cols, embed_cols, events = _get_scheme_data(scheme)
    if event not in events:
        result['status'] = 'event_not_found'
        return result

    event_df = filter_event_rows(full_df, event)
    if event_df.empty:
        result['status'] = 'skip_no_rows'
        return result

    all_feature_cols = BASE_VARS + type_cols + embed_cols
    label = f'{scheme}:{event}'
    try:
        event_df, dropped = validate_cox_inputs(event_df, event, f'tt_{event}', all_feature_cols, label=label)
    except ValueError as e:
        result['status'] = f'skip_validation: {e}'
        return result

    tc = [c for c in type_cols if c not in dropped]
    ec = [c for c in embed_cols if c not in dropped]
    afc = [c for c in all_feature_cols if c not in dropped]
    n_before = len(event_df)
    event_df = event_df.dropna(subset=afc + [event, f'tt_{event}'])
    if n_before - len(event_df):
        print(f'{label} dropped {n_before - len(event_df)}/{n_before} rows with NaN')

    # --- text model (grid search over l1_ratio x alpha) ---
    t0 = time.time()
    text_test, text_val, _ = run_grid_CoxPH_parallel(
        event_df, BASE_VARS + tc, ['AGE_AT_TREATMENTSTART'] + ec, ec,
        DEFAULT_L1_RATIOS, DEFAULT_ALPHAS,
        event_col=event, tstop_col=f'tt_{event}',
        max_iter=max_iter, n_jobs=n_jobs, backend=backend,
    )
    result['text_minutes'] = round((time.time() - t0) / 60, 2)

    # --- base model ---
    t0 = time.time()
    base_results = run_base_CoxPH(
        event_df, BASE_VARS + tc, ['AGE_AT_TREATMENTSTART'],
        event_col=event, tstop_col=f'tt_{event}', max_iter=max_iter,
    )
    result['base_minutes'] = round((time.time() - t0) / 60, 2)

    base_test = base_results[base_results['eval_data'] == 'test_data'].drop(columns='eval_data')
    base_val = base_results[base_results['eval_data'] == 'cv_data'].drop(columns='eval_data')

    new_base_c = float(base_test['mean_c_index'].iloc[0])
    new_text_c = float(text_test['mean_c_index'].iloc[0])
    new_delta = new_text_c - new_base_c
    result.update(new_base_c_index=new_base_c, new_text_c_index=new_text_c, new_delta_c_index=new_delta)

    # --- gate: overwrite only when the new delta beats the old delta ---
    improved = bool(pd.notna(old_delta) and (new_delta > old_delta))
    result['improved'] = improved
    if not improved:
        result['status'] = 'kept_original (no delta improvement)'
        return result

    out_dir = full_cohort_event_dir(scheme, event)
    os.makedirs(out_dir, exist_ok=True)
    filenames = ['text_test.csv', 'text_val.csv', 'base_test.csv', 'base_val.csv']
    if BACKUP_ORIGINALS:
        result['backup_dir'] = _backup_existing(scheme, event, out_dir, filenames)

    text_test.to_csv(os.path.join(out_dir, 'text_test.csv'), index=False)
    text_val.to_csv(os.path.join(out_dir, 'text_val.csv'), index=False)
    base_test.to_csv(os.path.join(out_dir, 'base_test.csv'), index=False)
    base_val.to_csv(os.path.join(out_dir, 'base_val.csv'), index=False)
    result['status'] = 'overwritten (delta improved)'
    return result

## 3. Re-run each negative-delta event

Iterates over the events selected in step 1 (most-harmful delta first). Each call re-fits both
models and overwrites the originals **only** if the new delta improves. Set `MAX_EVENTS` to a small
number first to sanity-check timing, then set it back to `None` for the full set.

In [ ]:
MAX_EVENTS = None  # e.g. 2 for a quick test; None = run all negative-delta events

to_run = negative_delta if MAX_EVENTS is None else negative_delta.head(MAX_EVENTS)
print(f'Re-running {len(to_run)} event(s)\n')

rerun_results = []
for i, row in to_run.iterrows():
    scheme, event = row['scheme'], row['event']
    print(f"[{i + 1}/{len(to_run)}] {scheme}:{event}  (old delta={row['delta_c_index']:+.4f})")
    try:
        res = rerun_if_improved(
            scheme, event,
            old_base_c=row['base_cindex'],
            old_text_c=row['text_cindex'],
        )
    except Exception as e:  # keep going even if one event blows up
        res = {'scheme': scheme, 'event': event, 'status': f'ERROR: {type(e).__name__}: {e}',
               'old_base_c_index': row['base_cindex'],
               'old_text_c_index': row['text_cindex'],
               'old_delta_c_index': row['delta_c_index']}
    rerun_results.append(res)
    print(f"    -> {res['status']}"
          + (f"  new delta={res['new_delta_c_index']:+.4f}" if pd.notna(res.get('new_delta_c_index', np.nan)) else ''))

rerun_df = pd.DataFrame(rerun_results)
print('\nStatus counts:')
print(rerun_df['status'].value_counts().to_string())

## 4. Old vs new delta comparison

Side-by-side view of the prior Fig-2 delta vs the re-run delta, flagging which events were
overwritten (delta improved) and which kept their original results. The record is saved to
`figure_data/negative_delta_rerun_<stamp>.csv`.

If any results were overwritten, re-run `prep_figure_2.py` to refresh `fig2_full_cohort_metrics.csv`
(and any downstream Figure 2 panels) with the new values.

In [ ]:
summary_cols = [
    'scheme', 'event', 'status', 'improved',
    'old_base_c_index', 'new_base_c_index',
    'old_text_c_index', 'new_text_c_index',
    'old_delta_c_index', 'new_delta_c_index',
    'text_minutes', 'base_minutes', 'backup_dir',
]
summary_cols = [c for c in summary_cols if c in rerun_df.columns]
comparison = rerun_df[summary_cols].copy()
if 'new_delta_c_index' in comparison and 'old_delta_c_index' in comparison:
    comparison['delta_change'] = comparison['new_delta_c_index'] - comparison['old_delta_c_index']

n_overwritten = int(rerun_df.get('improved', pd.Series(dtype=bool)).sum())
n_still_negative = int((rerun_df['new_delta_c_index'] < 0).sum()) if 'new_delta_c_index' in rerun_df else 0
print(f'Overwritten (delta improved): {n_overwritten} / {len(rerun_df)}')
print(f'Still negative after re-run : {n_still_negative} / {len(rerun_df)}')

# Persist the re-run record alongside the manuscript figure data
out_fp = os.path.join(FIGURE_DATA_DIR, f'negative_delta_rerun_{BACKUP_STAMP}.csv')
rerun_df.to_csv(out_fp, index=False)
print(f'Saved re-run record -> {out_fp}')

if n_overwritten:
    print('\nNOTE: results were overwritten. Re-run data_generation/prep_figure_2.py to refresh')
    print('      fig2_full_cohort_metrics.csv before regenerating Figure 2.')

float_cols = [c for c in comparison.columns if comparison[c].dtype.kind == 'f']
comparison.style.format({c: '{:+.4f}' for c in float_cols}) if hasattr(comparison, 'style') else comparison.round(4)